## Scanario 2

**TICKET:** "I set up a cron job to sync a directory to a backup location every 5 minutes. `crontab -l` shows it, and I can see in `journalctl` that cron is actually triggering it — but the backup folder is always empty. No errors reported anywhere."

**SYSTEM:** Ubuntu 22.04 LTS / WSL2

**START POINT:** You're logged in as `dennie`, sudo available. Nothing investigated yet — go.

**Break down**:
- a cron job is set up to **sync a directory** for every 5 minutes
- `crontab -l` & `journalctl` show it well
- the backup folder is empty **always**, NO errors reported

In [ ]:
dennie@HP:~$ crontab -l
*/5 * * * * cd project_backup && rsync -a . /home/dennie/backups/live/

In [ ]:
sudo journalctl -u cron -n 100 -p err..crit --no-pager
-- No entries --

Nothing at `err`/`crit` level from cron itself — makes sense, since cron considers its job "successfully launched" as long as it _executed_ the command; it has no idea whether `rsync` inside succeeded or failed.

Whatever's going wrong is happening **inside** the job, not in cron itself.

**Breaking down what actually happens inside cron**:

**Step 1 — `cd project_backup`**

This is a **relative path**. Under a normal interactive shell, `cd project_backup` works because your terminal's CWD is wherever you last `cd`'d to (e.g. `~/linux-tut`). But cron doesn't inherit _your_ shell's CWD:
> a user crontab job starts in the user's `$HOME` (`/home/dennie`), not wherever you happened to be when you ran `crontab -e`.

So cron actually tries: `cd /home/dennie/project_backup` — which almost certainly doesn't exist. `cd` fails and prints `No such file or directory` to stderr.

**Step 2 — the `&&`**

Because you chained with `&&`, the whole thing short-circuits: **if `cd` fails, `rsync` never runs at all.** Not "runs on the wrong directory" — doesn't run, period.

**Step 3 — where did the error go?**

This is the same MTA trap from your `hello.sh` lesson: cron captures the job's stdout/stderr and tries to mail it. No MTA installed → `journalctl`/syslog just logs "discarding output," and you never see the actual `cd: no such file or directory` message. That's exactly why journalctl shows cron _triggering_ the job successfully (it did — the shell ran) but nothing tells you the `cd` inside it failed.

**So to answer your question directly:** no — it does **not** successfully sync to `.../live/`. It _looks_ right because `rsync -a . /home/dennie/backups/live/` reads correctly in isolation ("archive-mode copy of current dir contents into `live/`, trailing slash on dest is fine"), but the `.` never means what you think — the `cd` that was supposed to put you in the right directory silently failed first, so that whole right-hand side of `&&` never executes.

**rsync -a itself, for the record:** `-a` = archive mode = recursive + preserves permissions, timestamps, symlinks, ownership, and skips already-up-to-date files. Correct flag choice for a backup job — not the bug.

**THE FIX** — remove the dependency on cron's CWD and capture output so this never hides again:

In [ ]:
*/5 * * * * rsync -a /home/dennie/linux-tut/project_backup/ /home/dennie/backups/live/ >> /home/dennie/backups/rsync.log 2>&1

> **Support rule:** any `cd relative/path && command` in a crontab is a red flag — cron's working directory is never guaranteed. Use absolute paths on both sides, and always redirect output so failures don't vanish into a nonexistent mail system.

That's an excellent, complete diagnosis — genuinely at the level I'd expect from a strong support engineer, not just someone reciting rules. A few things worth calling out about how well you nailed this:

- **The `&&` short-circuit insight** is the crux of the whole bug, and you explained _why_ it fails silently (stderr with nowhere configured to go, since there's no MTA in this WSL box and no redirect in the original line) rather than just "it's broken." That's exactly the reasoning a ticket write-up needs.
- **Correctly distinguished "runs on wrong directory" from "doesn't run at all"** — a common wrong instinct is to assume partial/wrong execution; you correctly identified `cd` failing means `rsync` _never even starts_.
- **Validated `-a` was already correct** instead of assuming everything in the broken line must be wrong — good discipline, don't fix what isn't broken.
- **Your fix is genuinely production-quality:** absolute paths eliminate the CWD dependency entirely, trailing slash on the _source_ (`project_backup/`) correctly means "copy contents into destination" rather than "create a nested `project_backup` folder inside `live/`" (a classic rsync trailing-slash gotcha you got right without me even prompting it), and `>> ... 2>&1` guarantees future failures land in a file you'll actually see instead of vanishing into an unconfigured mail queue.

One tiny polish for real-world practice, not a correction: after saving a fix like this, the fastest way to confirm it's _actually_ right before waiting 5 minutes for cron is to run the exact same command by hand first:

In [ ]:
rsync -a /home/dennie/linux-tut/project_backup/ /home/dennie/backups/live/
ls -l /home/dennie/backups/live/

If that works manually, cron running the identical absolute-path command will work too — cron behaving differently from a manual run is now specifically a `PATH`/environment issue, not a logic issue, which narrows any _future_ debugging fast.

**DEBRIEF — Scenario 2:**

- ✅ Went straight to `crontab -l` to read the exact scheduled command — right first move
- ✅ Correctly reasoned that `journalctl -u cron` empty at err/crit means the bug is _inside_ the job, not in cron itself — an important boundary to recognize
- ✅ Full, accurate root-cause explanation and a solid fix
- 🔁 Support rule locked in: **cron's working directory is always `$HOME`, never assume it's wherever you were when you edited the crontab — always use absolute paths in cron commands.**

Both scenarios closed out clean. That's Pillar 5 — concepts, practice, and scenarios — fully complete.